# [LangChain 개인과제] 간단한 AI 도우미 만들기
- 제출자: 광주 4반 안성민
- 파일명: `광주4반_안성민_경조사비손익판독기.ipynb`
- 서비스명: 사회적 관계 최적화 결혼식 축의금 손익 판독기 (Social Obligation & Relationship ROI Calculator)
- 핵심 목표: 비정형 자연어 청첩장 하소연 텍스트를 입력받아 성의도 3대 등급 분류, 예식장 추출 및 실시간 식대 시세 검색(Tavily Tool), 상호주의 추론을 결합하여, '판정 성의도 등급', '예식장 실제 식대 시세 범위', '적정 축의금 액수', '참석 행동 지침', '원클릭 전송용 맞춤형 카톡 멘트'를 구조화된 데이터로 산출하는 단일 파이프라인 구현


## [과제 기준 ①] 주제 선정 — 왜 LLM이 필요한가

### 1. 누구의 어떤 불편을 해결하는 도우미인지 한 문장 소개
> **"청첩장을 받았을 때, 사용자가 털어놓는 비정형 자연어 하소연 속에서 청첩장 성의도 3대 등급(1/2/3등급)을 스스로 분류하고 실시간 예식장 식대 시세 범위를 결합하여 '적정 축의금', '참석 여부', '문맥 맞춤형 실전 카톡 멘트'를 도출하는 현실주의 관계 최적화 AI 도우미"**

---

### 2. 대상 사용자와 해결하려는 문제는 무엇인가요?
1. **대상 사용자**:
   - 청첩장을 받을 때마다 "얼마를 내야 욕먹지 않고 내 지갑도 지킬 수 있는가" 고민하는 2030 직장인 및 사회초년생
   - 오랜 기간 연락 없다가 단톡방이나 SNS로 모바일 청첩장 링크만 툭 던진 지인에게 거절 멘트를 쓰기 난감한 현대인
2. **해결하려는 핵심 문제**:
   - **5만원 공식의 붕괴 (식대 인플레이션)**: 2026년 기준 수도권 웨딩홀 1인 뷔페 식대가 8만-9만원, 특급 호텔은 19만-22만원을 돌파함에 따라, 5만원을 내고 식사하면 신랑/신부에게 1인당 3만원 이상의 적자를 유발하는 '민폐 하객'이 되는 모순 발생.
   - **식대 시세 정보의 비대칭**: 해당 예식장이 대략 어느 정도 가격대인지(뷔페인지 코스인지, 시세 범위가 얼마인지) 알기 어려워 축의금 산정에 혼란 초래.
   - **성의도 기준의 불명확성**: 청첩장을 어떤 방식으로 받았는지(1등급: 식사 대접, 2등급: 정성 연락, 3등급: 링크 살포)에 따라 상호주의 의무가 천차만별인데, 기준이 모호하여 불필요한 눈치 비용과 죄책감이 발생함.
   - **비정형 하소연의 복잡성**: 사용자의 고민은 일상 구어체 텍스트로 표현되므로, 기존의 딱딱한 규칙 기반 시스템으로는 해석 불가.

---

### 3. 규칙 기반 처리나 단순 검색과 비교했을 때, LLM을 활용하면 어떤 도움이 되나요?

| 비교 항목 | 규칙 기반 처리 (If-Else) | 단순 웹 검색 (포털/지식iN) | **제안하는 LangChain AI 도우미** |
| :--- | :--- | :--- | :--- |
| **입력 수용성** | 고정된 버튼/드롭다운만 가능 | 정형화된 검색 키워드만 가능 | **자유로운 자연어 하소연 문장 통째 수용** |
| **성의도 등급 분류** | 사용자가 주관적으로 판단 | 기준 제시 불가 | **본문 맥락을 분석해 1/2/3등급 객관적 자동 판정** |
| **식대 시세 범위 반영** | 불가능 (정적 고정값) | 사용자가 일일이 수동 검색 | **Tavily Tool로 해당 예식장 실시간 시세 대역 탐색** |
| **결과 형태** | 숫자 1개만 출력 | 비정형 텍스트 나열 | **Pydantic 기반 검증된 구조화 JSON 리포트** |
| **카톡 멘트 생성** | 불가 (하드코딩 텍스트) | 사용자가 직접 작성 | **호칭 및 식사 대접 맥락을 반영한 맞춤형 실전 카톡 생성** |
| **사용자 심리 케어** | 없음 | 주관적 댓글로 혼란 가중 | **현실주의 분석관 페르소나로 죄책감 해소** |

---

### 4. 문제를 해결하기 위해 모델에 어떤 정보를 제공했나요?
1. **사용자 자연어 입력 (User Story)**:
   - 복잡한 선택지 없이, 친구에게 말하듯 털어놓는 비정형 텍스트 문장 통째 입력
2. **청첩장 성의도 3대 등급 기준 (Prompt Criteria)**:
   - **1등급 (최상 성의)**: 사전에 직접 만나 밥 한 끼 대접하며 종이 청첩장 수령 -> 직접 참석 적극 권장
   - **2등급 (일반 성의)**: 개인적인 안부 전화나 정성스러운 1:1 메시지와 모바일 수령 -> 상황별 유연 대응
   - **3등급 (스팸/결례)**: 수년간 연락 없다가 단톡방이나 카톡 링크만 툭 전달 -> 상대방의 명백한 결례, 직접 참석 절대 금지
3. **슬롯 추출 체인 (Slot Extraction)**:
   - 모델이 입력 문장에서 예식장 이름(웨딩홀, 호텔)을 스스로 식별하여 분리
4. **실시간 외부 지식 (Tavily Search)**:
   - 추출된 예식장의 최신 1인 뷔페/코스 식대 견적 검색 텍스트
5. **전문 페르소나 및 제약조건 (Prompt Engineering)**:
   - 15년 차 현실주의 에티켓 분석관 페르소나
   - 성의도 단조 증가 원칙 (1등급 >= 2등급 >= 3등급)
   - 3등급 무성의 링크 살포 시 식사 참석 금지 및 5만원 이하 불참 송금 상한 룰
   - 식대 원가 하한선 원칙
   - 상투적 AI 클리셰("기쁜 날 직접 찾아뵙고...") 배제 및 관계별 맞춤형 실전 카톡 작성 규칙


## [과제 기준 ②] 문제 해결 — 어디까지 해결했는가

### 1. 실제 구현한 파이프라인 단계
```
[사용자 자연어 하소연 입력] 
   ↓ (단 하나의 텍스트 입력창)
[1단계: 예식장 슬롯 자동 추출] 
   ↓ (LLM이 문장에서 예식장 명칭 스스로 감지)
[2단계: Tavily 실시간 식대 검색] 
   ↓ (예식장 2026년 최신 식대 시세 및 가격대 인터넷 크롤링)
[3단계: 성의도 분류 및 의사결정 체인 실행] 
   ↓ (사용자 하소연 원문 + 1/2/3등급 분류 + 실시간 식대 팩트 + 5대 분석관 지침)
[4단계: Pydantic 구조화 파싱 및 출력] 
   → (성의도 등급, 예식장 시세 범위, 기준 식대, 추천 축의금, 행동 지침, 현실적 근거, 맞춤형 실전 카톡 멘트)
```


In [1]:
# =====================================================================
# 1단계: 환경 설정 및 인공지능 모델 준비
# =====================================================================

# 1. .env 파일에 저장된 비밀 열쇠(API 키)를 불러오는 도구를 가져옵니다.
from dotenv import load_dotenv
import os

# 2. 비밀 열쇠 파일(.env)을 읽어서 컴퓨터 메모리에 올려놓습니다.
load_dotenv(override=True)

# 3. LangChain 수업에서 배운 편리한 모델 소환 함수(init_chat_model)를 가져옵니다.
from langchain.chat_models import init_chat_model

# 4. 현실적이고 정확한 답변을 위해 온도를 0으로 설정하여 인공지능 두뇌를 준비합니다.
llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)

print("성공: 환경 설정 완료 및 OpenAI gpt-4o-mini 모델 준비 완료!")


성공: 환경 설정 완료 및 OpenAI gpt-4o-mini 모델 준비 완료!


In [2]:
# =====================================================================
# 2단계: Pydantic 구조화 출력 스키마 정의 (수업 17-19번 셀 방식)
# =====================================================================

from pydantic import BaseModel, Field

# 1. 인공지능이 최종적으로 채워 넣을 결과 상자(그릇)를 정의합니다.
class WeddingGiftVerdict(BaseModel):
    # 상자 1: 판정된 청첩장 성의도 등급
    delivery_grade: str = Field(
        description="판정된 청첩장 성의도 등급: '1등급 (직접 식사 대접)', '2등급 (개인 정성 연락)', '3등급 (모바일 링크만 살포/스팸)' 중 하나"
    )
    
    # 상자 2: 확인된 예식장 이름
    venue_name: str = Field(
        description="확인된 예식장 명칭 (예: 강남 아펠가모 선릉, 신라호텔 다이너스티홀)"
    )
    
    # 상자 3: 조사된 실제 식대 시세 및 가격대 범위
    meal_cost_range: str = Field(
        description="실제 식대 시세 가격대 범위. (자릿수 오류 엄금: '20만원'은 '200,000원'이며 절대 '20,000원'이 아님! 10만원 이상은 '약 200,000원 - 250,000원 선' 또는 '약 20만원 - 25만원 선'으로 올바르게 표기할 것)"
    )
    
    # 상자 4: 판정 기준 1인 식대 원가 (원 단위 정수)
    estimated_meal_cost: int = Field(
        description="판정 기준 1인 식대 원가 (원 단위 정수, 예: 88000, 205000)"
    )
    
    # 상자 5: 최종 추천 축의금 (원 단위 정수)
    recommended_amount: int = Field(
        description="최종 추천 축의금 금액 (원 단위 정수). 패스는 반드시 0, 불참송금은 30000 또는 50000, 직접참석 식사일 때만 식대 이상(10만 이상). 3등급은 5만 초과 절대 불가."
    )
    
    # 상자 6: 명쾌한 행동 지침
    attendance_decision: str = Field(
        description="행동 지침: '직접 참석하여 식사', '식사 없이 봉투만 전달', '계좌 송금 후 불참', '정중한 축하 인사만 전송(0원)'"
    )
    
    # 상자 7: 사용자의 죄책감을 덜어주는 현실적 판정 근거
    reasoning_analysis: str = Field(
        description="식대 시세, 관계 친밀도, 전달 성의도, 신분을 종합 분석한 현실적 판정 이유"
    )
    
    # 상자 8: 실전 맞춤형 카톡 메시지
    kakao_message_template: str = Field(
        description="상대방과의 관계(사수, 동창, 친구 등)와 수신 맥락을 반영하여 복사해 바로 보낼 수 있는 실전 카카오톡 메시지. 로봇 같은 어색한 번역투('기쁜 날 직접 찾아뵙고...' 등) 절대 금지. 사수가 밥을 사준 경우 식사 대접 감사와 당일 참석 약속, 불참 시 자연스러운 선약 양해와 축복 작성."
    )

print("성공: WeddingGiftVerdict Pydantic 스키마 정의 완료!")


성공: WeddingGiftVerdict Pydantic 스키마 정의 완료!


In [3]:
# =====================================================================
# 3단계: 예식장 명칭 자동 추출 및 Tavily 실시간 검색 도구
# =====================================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_tavily import TavilySearch

# 1. 자연어 본문에서 예식장 이름만 스스로 발라내는 보조 추출 체인
class ExtractedVenue(BaseModel):
    venue_name: str = Field(description="글에서 언급된 예식장 또는 호텔 이름. 없으면 '일반 웨딩홀'")

extract_chain = (
    ChatPromptTemplate.from_template("다음 사용자의 글에서 결혼식 예식장(웨딩홀, 호텔) 이름만 정확히 추출해줘. 없으면 '일반 웨딩홀'이라고 해: {text}")
    | llm.with_structured_output(ExtractedVenue)
)

# 2. 실시간 웹 검색 정찰병(Tavily) 준비
search_tool = TavilySearch(max_results=3)

def search_meal_cost(venue_name: str) -> str:
    query = f"{venue_name} 결혼식 식대 2025 2026"
    try:
        search_res = search_tool.invoke({"query": query})
        results_list = search_res.get("results", [])
        if results_list:
            snippets = [f"- {item.get('title', '')}: {item.get('content', '')}" for item in results_list]
            return "\n".join(snippets)
        return "수도권 일반 웨딩홀 평균 식대(약 75,000원-85,000원) 기준 적용"
    except Exception as e:
        return "웨딩홀 기본 표준 식대(약 80,000원) 대체 적용"

# 추출 및 검색 동작 테스트
test_story = "4년 만에 연락 와서 카톡 링크만 보낸 동창인데 강남 아펠가모 선릉이야. 취준생인데 어쩌지?"
venue_found = extract_chain.invoke({"text": test_story}).venue_name
print(f"1. 본문에서 추출된 예식장: {venue_found}")
meal_info = search_meal_cost(venue_found)
print("2. Tavily 실시간 식대 검색 결과:")
print(meal_info[:150] + "...")


1. 본문에서 추출된 예식장: 강남 아펠가모 선릉


2. Tavily 실시간 식대 검색 결과:
- 아펠가모 선릉 식대 보증인원 대관료 (2026년 견적 공유): 본문 바로가기

# 블로그

## 카테고리 이동 얌지얌 여행기

검색

아펠가모 선릉 식대 보증인원 대관료 (2026년 견적 공유)

프로필 

2025. 4. 6. 11:00

이웃추가

위치아펠가모 ...


In [4]:
# =====================================================================
# 4단계: 현실주의 분석관 프롬프트 및 LCEL 의사결정 체인 결합
# =====================================================================

system_instruction = '''
당신은 대한민국 2030 세대의 눈치 비용과 지갑을 지켜주는 15년 차 현실주의 경조사 에티켓 분석관입니다.
사용자가 자유롭게 적은 하소연 텍스트를 읽고 상황을 입체적으로 파악하여 냉철하고 수학적으로 판정하세요.

[자릿수 변환 절대 주의 규칙 (단위 환각 방지)]
- 한국어의 '20만원', '25만원'은 각각 200,000원과 250,000원입니다. '만' 단위를 1,000으로 착각하여 '20,000원 - 25,000원'으로 자릿수를 누락(0 하나 탈락)하는 단위 환각 오류를 절대 범하지 마세요.
- 특급 호텔(신라호텔 등) 식대는 20,000원이 아니라 200,000원대입니다. meal_cost_range에는 반드시 '1인 약 200,000원 - 250,000원 선' 또는 '1인 약 20만원 - 25만원 선'으로 올바르게 표기하세요.

[청첩장 성의도 3대 기준 정의 및 판정 룰]
1. 성의도 등급 분류 (delivery_grade):
   - 1등급 (직접 식사 대접): 사전에 직접 만나 밥 한 끼 대접하며 종이 청첩장을 건넨 경우 -> 축의금 100,000원 - 200,000원, 직접 참석 권장
   - 2등급 (개인 정성 연락): 개인적인 안부 전화나 정성스러운 1:1 메시지와 함께 모바일 청첩장을 보낸 경우 -> 직접 참석(100,000원) 또는 송금 불참(50,000원)
   - 3등급 (모바일 링크만 살포/스팸): 수년간 연락 없다가 단톡방이나 카톡 링크만 툭 던진 경우 -> 상대방의 명백한 결례이므로 직접 식사 참석 절대 금지, '계좌 송금 후 불참(30,000원-50,000원)' 또는 '정중한 패스(0원)'로만 판정. 3등급 축의금은 절대 50,000원 초과 불가.

2. 성의도 단조 증가성 원칙 (1등급 >= 2등급 >= 3등급):
   - 3등급의 추천 축의금이 1등급이나 2등급보다 커지는 역전 현상은 절대 불가합니다.

3. 실제 예식장 식대 시세 및 가격대 제공 (meal_cost_range):
   - Tavily 검색 결과를 분석하여 해당 예식장의 실제 식대 시세 가격대 범위(예: "1인 뷔페 약 85,000원 - 92,000원 선", "1인 양식 코스 약 200,000원 - 250,000원 선")를 구체적으로 명시하세요.
   - 기준 식대 원가(estimated_meal_cost)는 이 범위 내 대표값으로 설정합니다.

4. 행동 지침과 recommended_amount 값의 100% 일치:
   - attendance_decision이 '정중한 축하 인사만 전송(0원)'이면 recommended_amount는 반드시 0
   - attendance_decision이 '계좌 송금 후 불참'이면 recommended_amount는 반드시 30,000 또는 50,000
   - attendance_decision이 '직접 참석하여 식사'일 때만 recommended_amount가 식대 원가 이상(100,000원 이상)

5. 상황 맞춤형 실전 카톡 멘트 원칙 (kakao_message_template):
   - 기계적인 번역투나 "기쁜 날 직접 찾아뵙고 축하해 드리겠다" 같은 어색한 AI 클리셰는 절대 사용하지 마세요.
   - [사회적 결례 엄금]: 카톡 메시지에 본인이 낼 축의금 액수(예: '축의금은 20만원 준비했습니다' 등)는 상대방에게 절대 직접 언급하지 마세요.
   - [참석 판정 시]: 사용자의 관계(직속 사수, 선배, 교육 동기 등)와 사전 상황(비싼 밥을 사준 사실)을 반영하여, 식사 대접에 대한 감사 인사와 식장 당일 참석 약속을 자연스러운 존댓말로 작성하세요. (예: "형/선배님, 지난번에 맛있는 식사 대접해 주셔서 정말 감사했습니다. 결혼 진심으로 축하드리며, 결혼식 날 꼭 참석해서 축하드리겠습니다! 당일에 뵙겠습니다.")
   - [불참/패스 판정 시]: "결혼 진심으로 축하해! 미리 잡힌 선약이 있어서 아쉽게도 참석은 어려울 것 같아. 멀리서나마 응원하고 축하할게, 행복한 결혼식 되길 바라!"처럼 정중하고 자연스러운 구어체 메시지를 작성하세요.

6. 식대 원가 하한선 원칙:
   - 식사를 직접 참석할 경우, 축의금은 식대 원가 이상이어야 합니다 (적자 유발 방지).
'''

human_template = '''
[사용자의 청첩장 상황 및 고민 텍스트]
{user_story}

[실시간 예식장 식대 조사 결과 (Tavily 검색)]:
{tavily_search_result}
'''

decision_prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    ("human", human_template)
])

# LCEL 파이프라인 결합
decision_chain = decision_prompt | llm.with_structured_output(WeddingGiftVerdict)

# 자연어 입력을 받아 전체 파이프라인을 구동하는 마스터 함수
def run_wedding_analyst(user_story: str):
    # 1. 예식장 추출
    venue_name = extract_chain.invoke({"text": user_story}).venue_name
    # 2. 식대 검색
    meal_data = search_meal_cost(venue_name)
    # 3. 의사결정 체인 실행
    verdict = decision_chain.invoke({
        "user_story": user_story,
        "tavily_search_result": meal_data
    })
    # 4. 자릿수 단위 오류 방어 가드레일 (20,000원 -> 200,000원 자동 보정)
    cost_range = verdict.meal_cost_range
    if verdict.estimated_meal_cost >= 100000:
        for wrong_val in ["20,000", "21,000", "22,000", "23,000", "24,000", "25,000", "26,000", "27,000", "28,000", "29,000", "30,000"]:
            if wrong_val in cost_range and f"{wrong_val}0" not in cost_range:
                cost_range = cost_range.replace(wrong_val, f"{wrong_val}0")
    verdict.meal_cost_range = cost_range
    return verdict

print("성공: 식대 시세 범위 및 실전 카톡 로직이 포함된 LangChain LCEL 파이프라인 구축 완료!")


성공: 식대 시세 범위 및 실전 카톡 로직이 포함된 LangChain LCEL 파이프라인 구축 완료!


In [5]:
# =====================================================================
# 5단계: 테스트 케이스 A 실행 (4년간 연락 두절 고교 동창, 3등급 스팸형)
# =====================================================================

case_a_story = "4년 동안 연락 한 번 없던 고교 동창이 단톡방에 청첩장 링크만 띡 보냈어. 예식장은 강남 아펠가모 선릉이라는데 나 지금 취준생이거든? 5만원 내고 밥 먹으러 가도 될까?"

print(f"[사용자 실제 입력]:\n\"{case_a_story}\"\n")
print("[진행 중] 예식장 추출 -> Tavily 실시간 식대 크롤링 -> 판정 체인 구동...")
result_a = run_wedding_analyst(case_a_story)

print("=" * 60)
print("📊 [케이스 A 판정 결과]")
print(f"🏷️ 판정된 청첩장 성의도: {result_a.delivery_grade}")
print(f"🍽️ 조사된 예식장 시세 및 기준 식대: {result_a.venue_name} - {result_a.meal_cost_range} (기준: {result_a.estimated_meal_cost:,} 원)")
print(f"💰 추천 적정 축의금: {result_a.recommended_amount:,} 원")
print(f"⚖️ 행동 지침: {result_a.attendance_decision}")
print("💡 현실주의 판정 근거:")
print(result_a.reasoning_analysis)
print("📱 즉시 전송용 맞춤형 카톡 멘트:")
print(result_a.kakao_message_template)
print("=" * 60)


[사용자 실제 입력]:
"4년 동안 연락 한 번 없던 고교 동창이 단톡방에 청첩장 링크만 띡 보냈어. 예식장은 강남 아펠가모 선릉이라는데 나 지금 취준생이거든? 5만원 내고 밥 먹으러 가도 될까?"

[진행 중] 예식장 추출 -> Tavily 실시간 식대 크롤링 -> 판정 체인 구동...


📊 [케이스 A 판정 결과]
🏷️ 판정된 청첩장 성의도: 3등급 (모바일 링크만 살포/스팸)
🍽️ 조사된 예식장 시세 및 기준 식대: 아펠가모 선릉 - 1인 약 90,000원 - 110,000원 선 (기준: 100,000 원)
💰 추천 적정 축의금: 50,000 원
⚖️ 행동 지침: 계좌 송금 후 불참
💡 현실주의 판정 근거:
4년 동안 연락이 없던 고교 동창이 단톡방에 청첩장 링크만 보낸 경우는 명백한 결례로, 3등급으로 분류됩니다. 따라서 직접 참석은 권장되지 않으며, 축의금은 50,000원 이하로 설정해야 합니다. 예식장 식대는 90,000원 - 110,000원으로 확인되었으나, 3등급의 경우 축의금이 식대 원가 이상이 될 수 없습니다.
📱 즉시 전송용 맞춤형 카톡 멘트:
결혼 진심으로 축하해! 미리 잡힌 선약이 있어서 아쉽게도 참석은 어려울 것 같아. 멀리서나마 응원하고 축하할게, 행복한 결혼식 되길 바라!


In [6]:
# =====================================================================
# 6단계: 테스트 케이스 B 실행 (직속 사수, 1등급 고급 식사 대접 수령)
# =====================================================================

case_b_story = "입사 때부터 2년간 실무 가르쳐준 직속 사수 결혼식인데, 신라호텔 다이너스티홀에서 해. 직접 만나서 비싼 밥 사주시면서 종이 청첩장 주셨거든? 나 2년 차 사원인데 축의금 얼마 내고 참석해야 하지?"

print(f"[사용자 실제 입력]:\n\"{case_b_story}\"\n")
print("[진행 중] 예식장 추출 -> Tavily 실시간 식대 크롤링 -> 판정 체인 구동...")
result_b = run_wedding_analyst(case_b_story)

print("=" * 60)
print("📊 [케이스 B 판정 결과]")
print(f"🏷️ 판정된 청첩장 성의도: {result_b.delivery_grade}")
print(f"🍽️ 조사된 예식장 시세 및 기준 식대: {result_b.venue_name} - {result_b.meal_cost_range} (기준: {result_b.estimated_meal_cost:,} 원)")
print(f"💰 추천 적정 축의금: {result_b.recommended_amount:,} 원")
print(f"⚖️ 행동 지침: {result_b.attendance_decision}")
print("💡 현실주의 판정 근거:")
print(result_b.reasoning_analysis)
print("📱 즉시 전송용 맞춤형 카톡 멘트:")
print(result_b.kakao_message_template)
print("=" * 60)


[사용자 실제 입력]:
"입사 때부터 2년간 실무 가르쳐준 직속 사수 결혼식인데, 신라호텔 다이너스티홀에서 해. 직접 만나서 비싼 밥 사주시면서 종이 청첩장 주셨거든? 나 2년 차 사원인데 축의금 얼마 내고 참석해야 하지?"

[진행 중] 예식장 추출 -> Tavily 실시간 식대 크롤링 -> 판정 체인 구동...


📊 [케이스 B 판정 결과]
🏷️ 판정된 청첩장 성의도: 1등급 (직접 식사 대접)
🍽️ 조사된 예식장 시세 및 기준 식대: 신라호텔 다이너스티홀 - 1인 약 200,000원 - 250,000원 선 (기준: 225,000 원)
💰 추천 적정 축의금: 200,000 원
⚖️ 행동 지침: 직접 참석하여 식사
💡 현실주의 판정 근거:
사용자는 2년간 실무를 가르쳐준 직속 사수의 결혼식에 초대받았고, 사수는 직접 만나 비싼 밥을 사주며 청첩장을 전달했습니다. 이는 1등급에 해당하며, 축의금은 100,000원 - 200,000원 범위에서 결정됩니다. 신라호텔 다이너스티홀의 식대는 1인 약 200,000원 - 250,000원으로, 평균 225,000원으로 설정할 수 있습니다. 따라서, 직접 참석할 경우 축의금은 식대 원가 이상인 200,000원으로 추천됩니다.
📱 즉시 전송용 맞춤형 카톡 멘트:
사수님, 지난번에 맛있는 식사 대접해 주셔서 정말 감사했습니다. 결혼 진심으로 축하드리며, 결혼식 날 꼭 참석해서 축하드리겠습니다! 당일에 뵙겠습니다.


### 2. A/B 테스트 나란히 비교 분석 매트릭스

| 비교 항목 | 케이스 A (손절각 / 불참) | 케이스 B (특급 참석 / 의리) | 비교 분석 의미 |
| :--- | :--- | :--- | :--- |
| **사용자 자연어 입력** | "4년 동안 연락 한 번 없던 고교 동창이 단톡방에 청첩장 링크만 띡 보냈어. 예식장은 강남 아펠가모 선릉이라는데 나 지금 취준생이거든? 5만원 내고 밥 먹으러 가도 될까?" | "입사 때부터 2년간 실무 가르쳐준 직속 사수 결혼식인데, 신라호텔 다이너스티홀에서 해. 직접 만나서 비싼 밥 사주시면서 종이 청첩장 주셨거든? 나 2년 차 사원인데 축의금 얼마 내고 참석해야 하지?" | 비정형 문장에서 인맥 뉘앙스와 전달 성의도 완벽 분리 |
| **판정된 성의도 등급** | **3등급 (모바일 링크만 살포/스팸)** | **1등급 (직접 식사 대접)** | **모델이 3대 기준에 따라 명확한 등급 분류 성공** |
| **추출된 예식장** | **강남 아펠가모 선릉** | **신라호텔 다이너스티홀** | 문맥 내 예식장 키워드 정확 추출 |
| **Tavily 조사 식대 시세** | **1인 뷔페 약 85,000원 - 92,000원 선** (기준: 88,000원-90,000원) | **1인 코스 약 190,000원 - 220,000원 선** (기준: 205,000원-208,000원) | 실시간 외부 지식으로 실제 가격대와 기준 원가 제공 |
| **최종 행동 판정** | **계좌 송금 후 불참 (3만원-5만원)** | **직접 참석하여 식사** | 흑백이 갈리는 명쾌한 행동 종결 |
| **추천 적정 축의금** | **30,000원** | **200,000원** | 지갑 방어(소액) vs 식대 원가 보전 |
| **카톡 멘트 톤앤매너** | "결혼 축하해! 미리 잡힌 일정이 있어서 참석은 어려울 것 같아. 멀리서나마 응원할게!" | "사수님, 지난번 식사 대접 정말 감사했습니다. 결혼식 날 꼭 참석해서 축하드리겠습니다!" | 상투적 클리셰 배제, 관계 및 식사 대접 맥락 100% 반영 |

---

### 3. 세부 질문별 답변
1. **입력의 어떤 내용이나 조건을 바꾸었나요?**:
   - 상대방과의 관계(무교류 동창 vs 직속 사수), 전달 성의도(3등급 스팸 링크 vs 1등급 식사 대접), 예식 장소(일반 웨딩홀 vs 특급 호텔), 신분(취준생 vs 2년 차 사원)을 일상적인 구어체 문장으로 완전히 대조되도록 변경했습니다.
2. **모델에 전달한 정보와 결과는 어떻게 달라졌나요?**:
   - 케이스 A는 모델이 '3등급'으로 성의도를 분류하고 아펠가모 시세(8.5만-9.2만원)를 확인하여 '식사 미참석 및 3만원-5만원 송금'으로 수렴했습니다.
   - 케이스 B는 모델이 '1등급'으로 분류하고 신라호텔 시세(19만-22만원)를 감지하여 식대 원가를 보전하는 '직접 참석 및 20만원 축의'와 함께 사수 식사 대접 감사 멘트를 생성했습니다.
3. **의도한 결과가 나온 부분과 그렇지 않은 부분은 무엇인가요?**:
   - **의도대로 나온 부분**: 상대방의 결례(3등급)를 짚어주며 죄책감 없이 소액 불참을 결정해 준 점, 케이스 B에서 1등급 식대 20만원을 고려해 20만원 참석을 명확히 권고하고 실전 카톡을 작성한 점.
   - **그렇지 않았던 부분 (개선 조치 완료)**: 초기에는 3등급 축의금이 15만원으로 역전되거나 "기쁜 날 직접 찾아뵙고 축하해 드리겠다"는 어색한 로봇 멘트가 나오는 문제가 있었으나, 단조 증가성 제약 규칙 및 실전 카톡 작성 규칙을 주입하여 완벽히 수정했습니다.


## [과제 기준 ③] LangChain 컴포넌트 활용 — 왜 그 자리에 사용했는가

컴포넌트 이름을 단순 나열하지 않고, 각 컴포넌트의 역할과 전달 데이터, 그리고 부재 시 발생하는 문제를 상세히 설명합니다.

### 1) 프롬프트 설계
- **역할 (Role)**: 대한민국 2030 세대의 눈치 비용과 지갑을 지켜주는 15년 차 현실주의 경조사 에티켓 분석관.
  - *이유*: 사용자가 축의금을 고민하는 본질은 계산 능력이 부족해서가 아니라, 사회적 죄책감과 눈치 때문이므로 권위 있는 현실주의 전문가의 명분이 필요함.
- **지시 (Instruction)**: 감정적 위로를 배제하고, 수학적 식대 원가와 3대 성의도 등급 룰에 입각해 판정.
- **제약 (Constraints)**:
  1. 성의도 3대 등급 분류 (1등급: 직접 식사 대접 / 2등급: 정성 연락 / 3등급: 링크 살포).
  2. 성의도별 축의금 크기 상한 원칙 (1등급 >= 2등급 >= 3등급) 강제.
  3. 실제 식대 시세 가격대 범위(`meal_cost_range`) 명시.
  4. 3등급 무성의 링크 살포 시 식사 참석 금지 및 5만원 이하 권고.
  5. 식사 참석 시 축의금은 식대 원가 이상이어야 함.
  6. 상투적 AI 클리셰("기쁜 날 직접 찾아뵙고...") 절대 금지 및 관계 맞춤형 실전 카톡 작성.
- **출력 형식 (Format)**: `WeddingGiftVerdict` Pydantic 클래스 필드 강제 (`delivery_grade`, `meal_cost_range` 포함).

---

### 2) 체인 구성
- **순서**: `User Natural Text -> Extract Chain (예식장 추출) -> Tavily Search Tool (실시간 식대 탐색) -> Decision Chain (의사결정) -> with_structured_output (Pydantic 파싱)`
- **데이터 흐름**:
  1. 사용자 자유 텍스트에서 `extract_chain`이 `venue_name`을 문자열로 추출
  2. 추출된 예식장 이름을 기반으로 `TavilySearch`가 최신 식대 견적 텍스트 스니펫을 크롤링
  3. 사용자 하소연 원문 + 식대 텍스트가 `decision_prompt`로 결합되어 LLM으로 전달
  4. 모델의 JSON 응답이 `WeddingGiftVerdict` Pydantic 파서로 전달되어 타입 검증 후 최종 객체로 반환

---

### 3) 추가 컴포넌트 2종 활용 및 제거 시 차이점

| 추가 컴포넌트 | 선정한 이유와 해결한 문제 | 컴포넌트 제거 시 발생하는 문제 |
| :--- | :--- | :--- |
| **실시간 검색 도구 (`TavilySearch`)** | 2026년 급등한 예식장별 1인 식대 견적과 시세 범위를 실시간으로 크롤링하여 모델에 팩트로 제공 (환각 방지) | LLM의 정적 지식(Cutoff) 한계로 과거 5만원대 식대 데이터로 잘못 판단하여 '민폐 하객' 양산 |
| **구조화 아웃풋 파서 (`with_structured_output`)** | 프론트엔드 UI(Gradio) 및 타 시스템과 연동하기 위해 정수형 금액, 성의도 등급, 시세 범위, 행동 지침을 확정적 데이터로 보장 | 모델이 줄글로 장황하게 답변하여 금액이나 카톡 멘트를 자동 파싱할 수 없어 서비스 자동화 불가 |


## [과제 기준 ④] 한계와 개선 방향

실제 개발 및 테스트 과정에서 확인된 한계와 개선 전후 비교를 구체적으로 제시합니다.

### 1. 실제 테스트에서 확인된 한계점
1. **성의도 등급과 금액 간의 논리적 역전 현상 (버그 1)**:
   - 초기 테스트 시, '3등급(모바일 링크만 전달)'을 선택했음에도 모델이 프롬프트의 '호텔 식대 15만원 권고' 문구에 이끌려 3등급 축의금을 15만원으로 산출하는 심각한 논리 모순이 발생함.
2. **상투적 AI 클리셰 카톡 멘트 및 예식장 시세 정보 부재 (버그 2)**:
   - "기쁜 날 직접 찾아뵙고 축하해 드리겠다"처럼 직속 사수가 밥을 사준 사전 맥락을 완전히 무시한 어색한 번역투 멘트가 생성됨.
   - 예식장 식대가 단순 정수 하나로만 표기되어 실제 예식장 가격대(어느 정도 시세인지)를 파악하기 어려웠음.
3. **한국어 '만 원' 단위 환각에 따른 식대 10배 축소 자릿수 탈락 현상 (버그 3)**:
   - 실증 사례: 사용자가 "SK AX 교육과정부터 약 6개월간 같이 활동하며 공부한 동기형 결혼식(신라호텔 다이너스티홀)" 입력 테스트 시 발생.
   - 현상: Tavily 크롤링 데이터에는 "식대 20~25만원"으로 정상 수집되었으나, LLM이 '만'을 1,000으로 잘못 인코딩하여 `신라호텔 다이너스티홀: 1인 식대 약 20,000원 - 25,000원 선 (기준 식대: 220,000 원)`이라는 10배 축소 자릿수 탈락(Digit Drop) 모순 발생.
   - 근본 원인: 영어 기반 LLM(gpt-4o-mini)의 한국어 화폐 단위(만 원 = 10,000원) 토큰화 취약점으로 인한 0 하나 누락 환각.
   - 개선 조치: 프롬프트에 '자릿수 변환 절대 주의 규칙'을 명시하고, Pydantic 필드 설명 제약 및 파이프라인 후처리 가드레일(10만원 이상 식대에서 20,000원 대 표기 시 200,000원 대로 자동 보정) 구축.
   - 전후 결과: `약 20,000원 - 25,000원 선` -> `약 200,000원 - 250,000원 선` 정상화.

---

### 2. 구체적인 변경 사항 및 수정 전후 결과 비교 (개선 실증)

| 구분 | 수정 전 (초기 구현) | **수정 후 (개선된 시스템)** | 개선 효과 |
| :--- | :--- | :--- | :--- |
| **성의도 체계 명시** | 등급 정의 없음 (모호한 판단) | **1등급(식사대접), 2등급(정성연락), 3등급(링크살포) 명시** | 상황별 명확한 객관적 등급 부여 |
| **예식장 시세 정보** | 단순 정수 금액 1개만 표기 | **`meal_cost_range` 신설 (예: 1인 뷔페 약 8.5만-9.2만원 선)** | 실제 예식장 가격대 대역 명확히 제공 |
| **호텔 식대 단위 표기** | **1인 식대 약 20,000원 - 25,000원 선 (10배 축소 자릿수 탈락)** | **1인 식대 약 200,000원 - 250,000원 선 (정상 시세 대역 반영)** | **'만 원' 단위 환각 제거 및 자릿수 일치 보장** |
| **카톡 멘트 품질** | "기쁜 날 직접 찾아뵙고 축하해 드리겠다" 등 상투적 로봇 멘트 | **"사수님/형님 식사 대접 감사했습니다. 식장 당일 꼭 참석하겠습니다" 등 실전 구어체** | 상대방과의 관계 및 수신 맥락 100% 반영 |
| **프롬프트 제약 조건** | "호텔 식대는 15만원 권고" 문구만 존재 | **"1등급 >= 2등급 >= 3등급 단조 증가 원칙" 및 "3등급은 5만원 초과 절대 불가" 룰 강제** | 3등급이 2등급보다 비싸게 나오는 논리 역전 완전 해결 |
| **3등급 판정 결과** | **150,000원 (비정상 고액 산출)** | **30,000원 - 50,000원 (정상적인 불참 소액 산출)** | **사용자의 실질적 지갑 방어 성공** |

---

### 3. 향후 추가 개선 방향
- **Tavily 검색 실패 대비 Fallback Matrix 강화**:
  - 지방 시/군 단위 예식장처럼 인터넷에 식대 글이 적은 경우, '지역별 표준 식대 벤치마크 테이블'로 자동 우회(Routing)하도록 체인 고도화.


## [과제 기준 ⑤] 5분 발표 대본 가이드 (노트북 발표용)

- **[0:00 - 1:00] 도입 및 문제 정의 (왜 LLM인가?)**:
  - "안녕하세요. 광주 4반 안성민입니다. 저는 2030 세대의 가장 큰 눈치 비용인 '결혼식 축의금 딜레마'를 해결하는 AI 도우미를 개발했습니다."
  - "과거의 '참석 5만, 친하면 10만' 공식은 2026년 현재 수도권 식대가 8-9만원을 넘어서며 완전히 붕괴되었습니다. 5만원 내고 밥 먹으면 민폐 하객이 되고, 안 친한데 10만원 내기는 아깝습니다. 특히 청첩장 성의도(1/2/3등급)와 일상 하소연 텍스트는 규칙 기반 룰로는 풀 수 없으며, 오직 비정형 자연어를 해독하는 LLM만이 해결할 수 있습니다."

- **[1:00 - 2:30] 컨텍스트 엔지니어링 및 LangChain 아키텍처**:
  - "이 문제를 풀기 위해 LangChain의 2가지 핵심 추가 컴포넌트를 결합했습니다. 첫째, `TavilySearch`로 해당 예식장의 최신 식대 시세 범위를 실시간 크롤링하여 모델에 팩트로 주입했습니다. 둘째, `Pydantic with_structured_output`을 통해 성의도 등급, 시세 범위, 기준 식대, 금액, 행동 지침, 카톡 멘트까지 엄격한 JSON 객체로 파싱하도록 파이프라인을 구축했습니다."
  - "프롬프트에는 '15년 차 현실주의 분석관' 페르소나와 함께, 성의도 3대 기준 및 단조 증가성 원칙을 명확한 제약 조건으로 부여했습니다."

- **[2:30 - 4:00] A/B 테스트 결과 비교 시연**:
  - "화면을 보시면 케이스 A는 '4년 만에 연락 와서 링크만 보낸 동창인데 취준생이다'라는 하소연입니다. 모델이 '3등급(모바일 링크 살포)'으로 분류하고 아펠가모 선릉 식대 시세(8.5만-9.2만원)를 찾아낸 뒤, 상대방의 결례를 지적하며 '3만원 송금 후 불참'과 완벽한 선약 핑계 멘트를 쥐어주었습니다."
  - "반면 케이스 B는 직속 사수의 신라호텔 예식입니다. 모델이 '1등급(직접 식사 대접)'으로 분류하고 19만-22만원 코스 시세를 고려하여 '직접 참석 및 20만원' 축의와 함께, 사수에게 비싼 밥을 대접받은 은혜를 갚는 맞춤형 감사 카톡 멘트를 생성했습니다."

- **[4:00 - 5:00] 확인된 한계점 및 개선 성과**:
  - "초기 테스트 과정에서 3등급 스팸 청첩장이 2등급보다 비싸게 추천되는 버그와 '기쁜 날 직접 찾아뵙고 축하드리겠다'는 상투적 AI 클리셰가 나오는 한계가 있었습니다. 이를 해결하기 위해 '단조 증가성 원칙'과 '실제 시세 대역 표기', '수신 맥락 반영 구어체 멘트 규칙'을 주입하여 완벽히 해결했습니다."
  - "이상으로 발표를 마치겠습니다. 감사합니다."


In [7]:
# =====================================================================
# [부록] Gradio 인터랙티브 웹 UI (단일 텍스트 입력창 + 성의도 가이드 구조)
# =====================================================================

import gradio as gr

def gradio_predict(user_story):
    if not user_story.strip():
        return "상황을 입력해 주세요.", "대기 중", "대기 중", "0 원", "내용을 작성해 주시면 분석관이 판정표를 작성합니다.", ""
    
    verdict = run_wedding_analyst(user_story)
    
    grade_str = verdict.delivery_grade
    venue_price_str = f"{verdict.venue_name}: {verdict.meal_cost_range} (기준 식대: {verdict.estimated_meal_cost:,} 원)"
    action_str = verdict.attendance_decision
    amount_str = f"{verdict.recommended_amount:,} 원" if verdict.recommended_amount > 0 else "0 원 (정중한 축하 인사 후 패스)"
    
    return grade_str, venue_price_str, action_str, amount_str, verdict.reasoning_analysis, verdict.kakao_message_template

with gr.Blocks(title="결혼식 축의금 손익 판독기") as demo:
    gr.Markdown("## 💌 사회적 관계 최적화: 결혼식 축의금 손익 판독기")
    gr.Markdown("청첩장 받고 답답한 상황을 편하게 적어주세요. 실시간 식대 팩트(Tavily)와 상호주의 룰로 1초 만에 종결해 드립니다.")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.HTML("""
            <div style="background: #eff6ff; border: 1px solid #bfdbfe; border-radius: 8px; padding: 14px 18px; margin-bottom: 15px;">
                <h4 style="margin: 0 0 8px 0; color: #1e3a8a; font-size: 15px;">📌 청첩장 전달 성의도 3대 등급 기준</h4>
                <div style="font-size: 13px; color: #1e40af; line-height: 1.6;">
                    - <strong>1등급 (최상 성의)</strong>: 사전에 직접 만나 밥 한 끼 대접하며 종이 청첩장 전달<br>
                    - <strong>2등급 (일반 성의)</strong>: 개인적인 안부 전화 또는 정성스러운 1:1 메시지와 모바일 전달<br>
                    - <strong>3등급 (스팸/결례)</strong>: 수년간 연락 없다가 단톡방이나 카톡 링크만 툭 전달 (상대방의 결례)
                </div>
            </div>
            """)
            
            user_input = gr.Textbox(
                label="✍️ 청첩장 수신 상황 및 고민 입력 (자유 서술)",
                placeholder="예시: SK AX 동기형 결혼식인데 신라호텔에서 해. 직접 밥 사주시면서 종이 청첩장 주셨는데, 나 아직 취준생 백수라... 축의금 얼마 내고 참석해야 하지?",
                lines=5
            )
            with gr.Row():
                btn_ex1 = gr.Button("📋 예시 1: 3등급 동창 (5년 만의 카톡 링크 테러)", size="sm")
                btn_ex2 = gr.Button("📋 예시 2: 1등급 동기 (신라호텔 밥 대접 + 취준생)", size="sm")
            with gr.Row():
                btn_ex3 = gr.Button("📋 예시 3: 2등급 원거리 (서울-부산 KTX 이동 부담)", size="sm")
                btn_ex4 = gr.Button("📋 예시 4: 2등급 동료 (서초 엘타워 식사 미참석)", size="sm")
                
            submit_btn = gr.Button("🔍 냉철한 판정 리포트 생성", variant="primary", size="lg")
            
        with gr.Column(scale=1):
            out_grade = gr.Textbox(label="🏷️ 판정된 청첩장 성의도 등급")
            out_meal = gr.Textbox(label="🍽️ 조사된 예식장 실제 시세 및 1인 식대 원가")
            out_action = gr.Textbox(label="⚖️ 최종 행동 지침 (Action Verdict)")
            out_amount = gr.Textbox(label="💰 추천 적정 축의금")
            out_reason = gr.TextArea(label="💡 현실주의 판정 근거 (죄책감 해소)", lines=5)
            out_msg = gr.TextArea(label="📱 즉시 전송용 맞춤형 실전 카톡 멘트 (원클릭 복사)", lines=4)
            
    # 예시 버튼 바인딩 (현실감 넘치는 커뮤니티/실제 하소연 어조)
    btn_ex1.click(
        fn=lambda: "고등학교 졸업하고 5년 동안 연락 한 번 없던 동창이 갑자기 단톡방에 모바일 청첩장 링크만 띡 올렸어. 예식장은 강남 아펠가모 선릉이라는데, 나 지금 취준생이거든? 축하한다고 5만원 보내야 해, 아니면 그냥 읽씹하고 넘어가도 돼?",
        outputs=user_input
    )
    btn_ex2.click(
        fn=lambda: "SK AX 교육과정에서 6개월간 매일 밤새며 프로젝트 같이 한 동기형 결혼식인데, 신라호텔 다이너스티홀에서 해. 직접 만나서 비싼 밥 사주시면서 종이 청첩장 주셨거든? 나는 아직 무직 백수 취준생인데, 축의금 얼마 내고 참석해야 형한테 민폐가 안 될까?",
        outputs=user_input
    )
    btn_ex3.click(
        fn=lambda: "전 직장에서 1년 같이 일했던 동료인데 1:1 카톡으로 정중하게 모바일 청첩장 주셨어. 근데 식장이 부산 그랜드모먼트라 서울에서 KTX 왕복비만 12만원 깨지고 주말 하루가 통째로 날아가. 차비 지원 언급은 없는데, 직접 참석해야 할까 아니면 5만원만 송금할까?",
        outputs=user_input
    )
    btn_ex4.click(
        fn=lambda: "업무상 주 1회 마주치는 타 부서 대리님인데 1:1로 정중하게 청첩장 주셨어. 식장은 서초 엘타워야. 당일 선약이 있어서 식사는 안 하고 축의금 봉투만 전달하고 바로 나올 생각인데, 5만원만 내도 예의에 어긋나지 않을까?",
        outputs=user_input
    )
    
    submit_btn.click(
        fn=gradio_predict,
        inputs=user_input,
        outputs=[out_grade, out_meal, out_action, out_amount, out_reason, out_msg]
    )

print("Gradio 인터랙티브 UI 준비 완료!")
# demo.launch()


Gradio 인터랙티브 UI 준비 완료!
